# Check beersheba tracks

- load in 1 ldc
- select events in ROI
- ensure they look normal

In [3]:

# import stuff
import sys,os,os.path
import csv
import traceback
#sys.path.append("../../")   # cite IC from parent directory
sys.path.append("/gluster/data/next/software/IC_311024/")
sys.path.append(os.path.expanduser('~/code/eol_hsrl_python'))
sys.path.append("/gluster/data/next/notebooks/")
os.environ['ICTDIR']='/gluster/data/next/software/IC_311024/'

from invisible_cities.core.core_functions   import shift_to_bin_centers
from invisible_cities.io.dst_io           import load_dst, load_dsts, df_writer

import FOM_functions as FOM_func
import functions_HE as func
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd





import scipy.special as special
from scipy.stats import norm
from scipy.stats import skewnorm, crystalball
from scipy.optimize import curve_fit
from tqdm import tqdm

from scipy.integrate import quad

import iminuit
from iminuit import Minuit
import probfit
from concurrent.futures import ProcessPoolExecutor

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import tables as tb
from matplotlib import colors 

from typing          import Optional
from typing          import Union
from typing          import Callable

from concurrent.futures import ProcessPoolExecutor

import sys,os,os.path
from pathlib import Path

from invisible_cities.io.dst_io           import load_dst, load_dsts, df_writer
from invisible_cities.io.hits_io          import hits_writer
from invisible_cities.core                import tbl_functions   as tbl
from invisible_cities.core.core_functions import in_range
#from invisible_cities.cities.beersheba    import hitc_to_df_
from invisible_cities.io.hits_io          import hits_from_df
from invisible_cities.evm.nh5             import HitsTable
from invisible_cities.types.symbols       import NormStrategy
from invisible_cities.types.ic_types      import NoneType
from invisible_cities.reco.corrections    import read_maps, get_df_to_z_converter, apply_all_correction
from invisible_cities.evm.event_model     import HitCollection

from tqdm import tqdm



In [ ]:
def load_in_data(FOM_TS):
    # load in the data
    CITY = 'beersheba'
    #FOM_TS = ['200326_15']
    #FOM_TS = ['201025']
    TIMESTAMP = FOM_TS
    RUN_NUMBER = 'th_port1a_dep_202602_subsample'
    # make directory
    pre_dir = '/gluster/data/next/files/TOPOLOGY_John/MC_data/'
    folder_name = f'{pre_dir}/{RUN_NUMBER}/{CITY}/{FOM_TS[0]}'
    folder_s = Path(f'{folder_name}')

    print(folder_s)

    tdst = []
    for LDC in tqdm(range(1,2)):
        files = list(Path(f'{folder_name}/ldc{LDC}/').rglob('*.h5'))
        for file in tqdm(files):
            try:
                df = load_dst(file, 'DECO', 'Events')
            except:
                traceback.format_exc()
                df = pd.DataFrame()
            tdst.append(df)

    tdst = pd.concat(tdst)
    display(tdst)
    return tdst

tdst = load_in_data(['250326_MC'])

/gluster/data/next/files/TOPOLOGY_John/MC_data/th_port1a_dep_202602_subsample/beersheba/250326_MC


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# load in one or two files
path = '/gluster/data/next/files/TOPOLOGY_John/MC_data/th_port1a_dep_202602_subsample/beersheba/250326_MC'
path = list(Path(f'{path}/ldc1/').rglob('*.h5'))
full_data = []
tracks = []
for i in range(5):
    data = load_dst(path[i], 'DECO', 'Events')
    for j, df in data.groupby('event'):
        if ((df.Ec.sum() > 1.4) & (df.Ec.sum() < 1.8)):
            full_data.append(data)

    tracks.append(pd.concat(full_data))
    full_data = []

tracks = pd.concat(tracks)

,event,npeak,X,Y,Z,E,Xpeak,Ypeak,time,nsipm,Xrms,Yrms
0,150100000,0.0,145.875,220.575,583.144089,9.151333e-09,166.561969,220.780157,3.002000e+14,0,0,0
1,150100000,0.0,145.875,221.575,583.144089,1.682047e-08,166.561969,220.780157,3.002000e+14,0,0,0
2,150100000,0.0,145.875,222.575,583.144089,2.304515e-08,166.561969,220.780157,3.002000e+14,0,0,0
3,150100000,0.0,145.875,223.575,583.144089,2.355698e-08,166.561969,220.780157,3.002000e+14,0,0,0
4,150100000,0.0,145.875,224.575,583.144089,1.824812e-08,166.561969,220.780157,3.002000e+14,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9108525,150100168,1.0,284.875,277.575,686.002667,7.343238e-09,273.183252,268.831377,3.002003e+14,0,0,0
9108526,150100168,1.0,284.875,278.575,686.002667,1.055458e-08,273.183252,268.831377,3.002003e+14,0,0,0
9108527,150100168,1.0,284.875,279.575,686.002667,9.828445e-09,273.183252,268.831377,3.002003e+14,0,0,0
9108528,150100168,1.0,284.875,280.575,686.002667,5.996122e-09,273.183252,268.831377,3.002003e+14,0,0,0


KeyboardInterrupt: 